In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

import keras
from keras import layers, regularizers
from keras.callbacks import EarlyStopping, ModelCheckpoint
from bidict import bidict
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [ ]:
ENCODER = bidict({
    'a': 0,
    'o_u': 1, 
    'e_i': 2,
})

In [ ]:
labels = np.load("../data_v_1/labels.npy")
labels = labels - 1
labels.shape

In [ ]:
imgs = np.load("../data_v_1/imgs.npy").astype("float32") / 255

In [ ]:
# check out one of the images
plt.figure()
plt.imshow(imgs[0])
plt.grid(False)
plt.show()

In [ ]:
if imgs.ndim == 3:
    imgs = np.expand_dims(imgs, -1)

In [ ]:
imgs_train, imgs_test, labels_train, labels_test = train_test_split(
    imgs, labels, test_size=0.20, stratify=labels, random_state=42
)

In [ ]:
print(imgs_train.shape)
print(labels_train.shape)

print(labels_train.min())
print(labels_train.max())

In [ ]:
def add_noise(img):
    noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=0.05, dtype=tf.float32)
    return tf.clip_by_value(img + noise, 0.0, 1.0)

datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
)
datagen.fit(imgs_train)

In [ ]:
for x_batch, y_batch in datagen.flow(imgs_train, labels_train, batch_size=4):
    for i in range(4):
        plt.subplot(1, 4, i+1)
        plt.imshow(x_batch[i].squeeze(), cmap='gray')
        plt.title(f"Label: {y_batch[i]}")
        plt.axis('off')
    plt.show()
    break

In [ ]:
from tensorflow.keras import regularizers

batch_size = 8
epochs = 30

model = keras.Sequential([
    keras.Input(shape=(50, 50, 1)),

    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.BatchNormalization(),

    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.BatchNormalization(),

    layers.Flatten(),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(3, activation='softmax')
])

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=4, restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=2
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1,
    min_lr=1e-6
)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)

early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
# checkpoint = ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True)

optimizer = keras.optimizers.Adam()

model.compile(optimizer=optimizer,
              loss=loss_fn,
              metrics=['accuracy'])

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels_train),
    y=labels_train
)

class_weight_dict = dict(enumerate(class_weights))

model.fit(
    datagen.flow(imgs_train, labels_train, batch_size=batch_size),
    epochs=epochs,
    validation_data=(imgs_test, labels_test),
    callbacks=[early_stopping, lr_scheduler],
    class_weight=class_weight_dict
)

In [ ]:
print(labels_train[:5])  # should be like: [0, 2, 1, 0, 1]
print(np.unique(labels_train))  # should show: [0 1 2]

32 128-3-2 256-3-2 .9102
32 256-3-2 128-3-2 .9077
32 128-3-2 256-3-2 512-3-2 .9127
32 128-1-2 256-3-2 512-5-2 .9327
32 128-5-2 256-5-2 512-5-2 .9626

In [ ]:
model.evaluate(imgs_test, labels_test)

In [ ]:
y_pred = model.predict(imgs_test)
y_pred_labels = np.argmax(y_pred, axis=1)

cm = confusion_matrix(labels_test, y_pred_labels)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=ENCODER.inverse, yticklabels=ENCODER.inverse)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
wrong = np.where(y_pred_labels != labels_test)[0]
for i in wrong[:5]:
    plt.imshow(imgs_test[i].squeeze(), cmap='gray')
    print(f"True: {ENCODER.inverse[labels_test[i]]}, Pred: {ENCODER.inverse[y_pred_labels[i]]}")
    plt.show()

In [ ]:
import numpy as np

sample_imgs = imgs_test[:5]
sample_labels = labels_test[:5]

preds = model.predict(sample_imgs)
predicted_classes = np.argmax(preds, axis=1)

print("Predicted:", predicted_classes)
print("Actual:   ", sample_labels)

In [ ]:
import numpy as np
from collections import Counter

print("Train:", Counter(labels_train))
print("Test:", Counter(labels_test))

In [ ]:
y_pred = model.predict(imgs_test).argmax(axis=1)
cm = confusion_matrix(labels_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
model.summary()

In [ ]:
model.save("model_v1_2.keras")